# Kuramoto Benchmark

This notebook imports saved controlled noisy Kuramoto runs from `results/kuramoto`. It is analysis-only and does not launch training.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from mfc.visualization import (
    best_runs_by_label,
    flow_dataframe,
    load_env_and_policy,
    load_runs,
    objective_table,
    plot_state_flow,
    plot_validation_rewards,
    runtime_table,
)
from mfc.visualization.io import run_label

ENV = 'kuramoto'
RESULTS_ROOT = ROOT / 'results'
runs = load_runs(RESULTS_ROOT, env=ENV)
print(f'Loaded {len(runs)} saved runs from {RESULTS_ROOT / ENV}')

## Validation Reward

Mean validation reward over training, with one standard deviation across seeds. The planned Kuramoto runs use particle population flow.

In [ ]:
if not runs:
    print('No saved runs yet. Run scripts/run.py or scripts/parallel_train.py before executing the analysis cells.')
else:
    for horizon in sorted({run['metadata']['horizon'] for run in runs}):
        fig, ax = plt.subplots(figsize=(8, 4.5))
        try:
            plot_validation_rewards(runs, env=ENV, horizon=horizon, flow='particle', ax=ax)
            ax.set_title(f'Kuramoto validation reward, T={horizon}')
            plt.show()
        except ValueError as exc:
            plt.close(fig)
            print(exc)

## Fourier Moment Flow

The learned population law is summarized by the first Fourier moments `C`, `S`, and the order parameter `R = sqrt(C^2 + S^2)`. The plots below use the best saved seed for each algorithm and perturbation setting.

In [ ]:
if not runs:
    print('No saved runs yet.')
else:
    for run in best_runs_by_label(runs):
        fig, ax = plt.subplots(figsize=(8, 4.5))
        plot_state_flow(run, ax=ax)
        meta = run['metadata']
        label = run_label(meta)
        ax.set_title(f'{label}, flow={meta["flow"]}')
        plt.show()

## Order Parameter Table

Final and maximum order parameter reached by each selected policy. Larger values indicate stronger synchronization of the population.

In [ ]:
rows = []
for run in best_runs_by_label(runs):
    meta = run['metadata']
    flow = flow_dataframe(run)
    if flow.empty:
        continue
    rows.append({
        'label': run_label(meta),
        'horizon': meta['horizon'],
        'flow': meta['flow'],
        'seed': meta['seed'],
        'final_R': flow['order_parameter'].iloc[-1],
        'max_R': flow['order_parameter'].max(),
        'final_C': flow['cos_moment'].iloc[-1],
        'final_S': flow['sin_moment'].iloc[-1],
    })
order_table = pd.DataFrame(rows)
display(order_table)

## Phase Cloud Rollout

A qualitative rollout for one saved policy. Points are particles on the unit circle at selected times; the red marker shows the mean direction and length of the order parameter.

In [ ]:
def rollout_phase_cloud(run, n_particles=512, seed=1234):
    env, policy = load_env_and_policy(run)
    generator = torch.Generator(device=env.device)
    generator.manual_seed(seed)
    states = env.sample_initial(n_particles, generator)
    snapshots = {0: states.detach().cpu()}
    horizon = run['metadata']['horizon']
    selected = {0, horizon // 2, horizon}

    with torch.no_grad():
        for t in range(horizon):
            law = env.empirical_law(states)
            actions = env.sample_action(policy, t, states, law, generator)
            states = env.sample(states, law, actions, generator, t=t)
            if t + 1 in selected:
                snapshots[t + 1] = states.detach().cpu()
    return snapshots

if not runs:
    print('No saved runs yet.')
else:
    run = best_runs_by_label(runs)[0]
    snapshots = rollout_phase_cloud(run)
    fig, axes = plt.subplots(1, len(snapshots), figsize=(4.2 * len(snapshots), 4.2), subplot_kw={'aspect': 'equal'})
    if len(snapshots) == 1:
        axes = [axes]
    for ax, (time, phases) in zip(axes, snapshots.items()):
        x = torch.cos(phases).numpy()
        y = torch.sin(phases).numpy()
        ax.scatter(x, y, s=8, alpha=0.35)
        c = x.mean()
        s = y.mean()
        ax.arrow(0, 0, c, s, width=0.015, color='tab:red', length_includes_head=True)
        circle = plt.Circle((0, 0), 1.0, fill=False, color='black', alpha=0.35)
        ax.add_artist(circle)
        ax.set_xlim(-1.1, 1.1)
        ax.set_ylim(-1.1, 1.1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f't={time}')
    fig.suptitle(run_label(run['metadata']))
    plt.show()

## Objective and Runtime Tables

Final validation reward, estimated simulator budget, and wall-clock timing for each saved configuration.

In [ ]:
display(objective_table(runs))
display(runtime_table(runs))